## Create a new environment in bioconda (miniconda)

In [ ]:
!conda create -n 16s-nanopore -c bioconda -c conda-forge nanoplot cutadapt chopper kma emu osfclient -y
!conda activate 16s-nanopore

In [ ]:
# Sanity check using 'conda run' to point to the correct environment
!conda run -n 16s-nanopore NanoPlot --version
!conda run -n 16s-nanopore cutadapt --version
!conda run -n 16s-nanopore chopper --version
!conda run -n 16s-nanopore emu --version
!conda run -n 16s-nanopore osf -V
!conda run -n 16s-nanopore kma -v

## Renamed

In [1]:
import os
import shutil
import pandas as pd

# 1. Paths
folder = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/its'
csv_file = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/meta_date_its.csv'
renamed_folder = os.path.join(folder, 'renamed')  # raw files stay untouched
os.makedirs(renamed_folder, exist_ok=True)

# 2. Read metadata (comma is the expected separator for a .csv)
df = pd.read_csv(csv_file, sep=',')
assert {'Name', 'sample_title'}.issubset(df.columns), f"Missing columns, found: {df.columns.tolist()}"

dupes = df['sample_title'][df['sample_title'].duplicated()].tolist()
if dupes:
    raise ValueError(f"Duplicate sample_title values would overwrite files: {dupes}")

log = []

# 3. Copy files under the new name, preserving the original extension
for _, row in df.iterrows():
    old_name = str(row['Name']).strip()
    sample_title = str(row['sample_title']).strip()
    old_path = os.path.join(folder, old_name)

    if not os.path.exists(old_path):
        print(f"[ERROR] File not found: {old_name}")
        continue

    if old_name.endswith('.fastq.gz'):
        ext = '.fastq.gz'
    elif old_name.endswith('.fastq'):
        ext = '.fastq'
    else:
        ext = os.path.splitext(old_name)[1]

    new_name = f"{sample_title}{ext}"
    new_path = os.path.join(renamed_folder, new_name)

    shutil.copy2(old_path, new_path)
    log.append({'old_name': old_name, 'new_name': new_name})
    print(f"[SUCCESS] Copied: {old_name} -> {new_name}")

pd.DataFrame(log).to_csv(os.path.join(renamed_folder, 'rename_manifest.csv'), index=False)
print("Done!")

[SUCCESS] Copied: Controle-1_ITS.fastq -> CTL-1.fastq
[SUCCESS] Copied: Controle-2_ITS.fastq -> CTL-2.fastq
[SUCCESS] Copied: Controle-3_ITS.fastq -> CTL-3.fastq
[SUCCESS] Copied: Desn-1_ITS.fastq -> DES-1.fastq
[SUCCESS] Copied: Desn-2_ITS.fastq -> DES-2.fastq
[SUCCESS] Copied: Desn-3_ITS.fastq -> DES-3.fastq
[SUCCESS] Copied: M-Desn-1_ITS.fastq -> MOT-1.fastq
[SUCCESS] Copied: M-Desn-2_ITS.fastq -> MOT-2.fastq
[SUCCESS] Copied: M-Desn-3_ITS.fastq -> MOT-3.fastq
[SUCCESS] Copied: P-Desn-1_ITS.fastq -> FAT-1.fastq
[SUCCESS] Copied: P-Desn-2_ITS.fastq -> FAT-2.fastq
[SUCCESS] Copied: P-Desn-3_ITS.fastq -> FAT-3.fastq
Done!


## Library and directory preparation

In [2]:
import os
import subprocess

# 1. Paths configuration (Updating based on your previous paths)
renamed_folder = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/its/renamed'
base_out = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma'

# Define specific output folders for each step
qc_dir = os.path.join(base_out, '1_qc_raw')
trimmed_dir = os.path.join(base_out, '2_trimmed')
filtered_dir = os.path.join(base_out, '3_filtered')
emu_dir = os.path.join(base_out, '4_emu_taxa')

# Create directories if they don't exist
for directory in [qc_dir, trimmed_dir, filtered_dir, emu_dir]:
    os.makedirs(directory, exist_ok=True)

# 2. Get list of renamed files
fastq_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]
print(f"Found {len(fastq_files)} fastq files to process.")

Found 12 fastq files to process.


## Quality control of the reads

In [ ]:
from pathlib import Path
import subprocess

env = "16s-nanopore"
threads = "4"

renamed_folder = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/its/renamed")
qc_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/1_qc_raw")
qc_dir.mkdir(parents=True, exist_ok=True)

subprocess.run(
    f"conda install -n {env} -c bioconda -c conda-forge nanocomp -y",
    shell=True,
    check=True
)

fastq_files = sorted(
    file for file in renamed_folder.iterdir()
    if file.name.endswith((".fastq", ".fastq.gz"))
)

print(f"Starting QC for {len(fastq_files)} samples...")

for file in fastq_files:
    sample = file.name.split(".fastq")[0]
    print(f"Running NanoPlot (with dot plots): {sample}")

    subprocess.run([
        "conda", "run", "-n", env, "NanoPlot",
        "-t", threads, "--fastq", str(file),
        "-o", str(qc_dir / sample),
        "--plots", "dot"
    ], check=True)

nanocomp_out = qc_dir / "NanoComp_Report"
nanocomp_out.mkdir(exist_ok=True)

print("Generating NanoComp report...")
subprocess.run([
    "conda", "run", "-n", env, "NanoComp", "-t", threads,
    "--fastq", *map(str, fastq_files),
    "--names", *(file.name.split(".fastq")[0] for file in fastq_files),
    "-o", str(nanocomp_out)
], check=True)

print("\nQuality control completed!")
print(f"NanoComp report: {nanocomp_out}")

2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - bioconda
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done




==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.7.1

Please update conda by running

    $ conda update -n base -c conda-forge conda





## Package Plan ##

  environment location: /home/marcos/miniconda3/envs/16s-nanopore

  added / updated specs:
    - nanocomp


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    openssl-3.6.4              |       h781a0a9_0         3.1 MB  conda-forge
    psutil-7.2.2               |  py313hd42f317_2         223 KB  conda-forge
    wrapt-2.4.0                |  py313hd42f317_1         141 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         3.4 MB

The following NEW packages will be INSTALLED:

  deprecated         conda-forge/noarch::deprecated-1.3.1-pyhd8ed1ab_1 
  nanocomp           bioconda/noarch::nanocomp-1.25.6-pyhdfd78af_0 
  nanomath           bioconda/noarch::nanomath-1.4.0-pyhdfd78af_0 
  psutil             conda-forge/linux-64::psutil-7.2.2-py313hd42f317_2 
  wrapt              conda-forge/l

For decision trimmer, use the LengthvsQualityScatterPlot_kde, which is a tool that generates a scatter plot of read lengths versus quality scores. This can help identify any issues with the sequencing data, such as low-quality reads or unexpected length distributions.

## Filter reads by length with chopper

Trim the reads to a specific length range using chopper. This step is important to ensure that only reads of the desired length are retained for downstream analysis. The command will filter the reads based on the specified minimum and maximum length thresholds.

In [ ]:
# 5. Run Chopper for quality and length filtering
import os
import subprocess

renamed_folder = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/its/renamed")
filtered_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/3_filtered")
filtered_dir.mkdir(parents=True, exist_ok=True)

fastq_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]

failed_samples = []

for file in fastq_files:
    sample_name = file.split('.fastq')[0]
    input_path = os.path.join(renamed_folder, file)
    output_path = os.path.join(filtered_dir, f"{sample_name}_filtered.fastq")
    
    bash_cmd = (
        f"conda run -n 16s-nanopore bash -c "
        f"\"zcat -f '{input_path}' | "
        f"chopper "
        f"-q 10 "
        f"--minlength 300 " #Is the minimum length of the reads to keep. Reads shorter than this will be discarded.
        f"--maxlength 1000 " #Is the maximum length of the reads to keep. Reads longer than this will be discarded.
        f"> '{output_path}'\""
    )
    
    print(f"Filtering {sample_name}...")
    
    try:
        # check=True Makes subprocess.run raise an exception if the command fails
        subprocess.run(bash_cmd, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[ERROR] {sample_name}!")
        print("Skipping to the next sample...\n")
        failed_samples.append(sample_name)
        
        if os.path.exists(output_path):
            os.remove(output_path)

print("\n--- Filtered end ---")
if failed_samples:
    print(f"Failed samples: {failed_samples}")
else:
    print("All samples were processed successfully!")

# Download UNITE database

In [ ]:
import os
import subprocess

# Path to the output directory where the UNITE database will be stored
base_out = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma'
unite_db_dir = os.path.join(base_out, 'unite_database')
os.makedirs(unite_db_dir, exist_ok=True)

tar_path = os.path.join(unite_db_dir, 'unite-fungi.tar')

# Path to the OSF storage for the UNITE database
osf_path = 'osfstorage/emu-prebuilt/unite-fungi.tar'

# Check if the taxonomy.tsv file exists to determine if the database has already been downloaded
check_file = os.path.join(unite_db_dir, 'taxonomy.tsv')

if not os.path.exists(check_file):
    print("Downloading the UNITE database...")
    
    # Download
    dl_cmd = f"conda run -n its-nanopore osf -p 56uf7 fetch '{osf_path}' '{tar_path}'"
    subprocess.run(dl_cmd, shell=True, check=True)
    
    print("Extracting the UNITE database...")
    # Extração
    tar_cmd = f"tar -xvf '{tar_path}' -C '{unite_db_dir}'"
    subprocess.run(tar_cmd, shell=True, check=True)
    
    print("Database downloaded and extracted successfully!")
else:
    print(f"The UNITE database already exists in {unite_db_dir}.")

# Generate a database for kma

1. Generate KMA database

In [ ]:
!mkdir -p /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/unite_db

# its
!conda run -n 16s-nanopore kma index -i /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/unite_database/species_taxid.fasta -o /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/unite_db

!echo "Database indexing completed successfully."

2. KMA classification (UNITE)

In [ ]:
#3.kma classification
import subprocess
from pathlib import Path

input_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/3_filtered")
kma_out = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/4_kma_taxa")
kma_out.mkdir(parents=True, exist_ok=True)

# Replace with your actual KMA database path
kma_db = "/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/unite_db/unite_db"

fastq_files = [f for f in input_dir.iterdir() if f.name.endswith('.fastq')]

print(f"Starting KMA classification for {len(fastq_files)} samples...")

for file in fastq_files:
    sample_name = file.name.split('_filtered')[0]
    out_prefix = kma_out / sample_name
    
    cmd = [
        "conda", "run", "-n", "16s-nanopore",
        "kma",
        "-i", str(file),
        "-o", str(out_prefix),
        "-t_db", kma_db,
        "-bcNano",         # Optimizes for Nanopore errors
        "-ont", # Optimizes for ONT reads
        "-ID", "98.0",   # Reliability: Minimum 98% identity
        "-p", "0.01",    # Reliability: 99% statistical confidence
        "-t", "4"
    ]
    
    print(f"Running KMA for {sample_name}...")
    
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        print(f"[ERROR] Failed to classify {sample_name}")

print("\nKMA classification finished.")

# Merge KMA results

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Define the directory where KMA saved the results
kma_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/4_kma_taxa")

# 2. Find all KMA result files (.res)
res_files = list(kma_dir.glob("*.res"))
print(f"Found {len(res_files)} KMA result files. Merging them now...")

df_list = []

# 3. Read each file and extract the taxonomy and abundance score
for file in res_files:
    sample_name = file.stem  # Gets the sample name without the .res extension
    
    # Read the tab-separated KMA output
    df = pd.read_csv(file, sep='\t')
    
    # Keep only the Taxonomy Name (#Template) and the Abundance (Score)
    df = df[['#Template', 'Score']]
    df = df.rename(columns={'Score': sample_name})
    df.set_index('#Template', inplace=True)
    
    df_list.append(df)

# 4. Combine all samples into a single matrix and fill missing values with 0
otu_matrix = pd.concat(df_list, axis=1).fillna(0)

# 5. Save the final matrix
output_file = kma_dir / "otu_abundance_matrix.csv"
otu_matrix.to_csv(output_file)

print(f"Matrix successfully saved to:\n{output_file}")

1. kda classification 

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Define paths
matrix_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/4_kma_taxa")
raw_matrix_path = matrix_dir / "otu_abundance_matrix.csv"
clean_matrix_path = matrix_dir / "otu_abundance_matrix_CLEAN.csv"

# Path to the taxonomy mapping file inside your database folder
tax_db_path = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/unite_database/taxonomy.tsv")

# 2. Load the raw KMA matrix
df = pd.read_csv(raw_matrix_path)

# Extract just the tax_id from the long KMA template string
df['tax_id'] = df['#Template'].apply(lambda x: str(x).split(':')[0])
df = df.rename(columns={'#Template': 'raw_taxonomy'})

print("Loading taxonomy database dictionary...")

try:
    # 3. Load the full taxonomy database
    # Ensures tax_id is read as a string to match properly
    tax_db = pd.read_csv(tax_db_path, sep='\t', dtype={'tax_id': str})
    
    # 4. Merge (Join) the abundance matrix with the full taxonomy using the tax_id
    df_merged = pd.merge(df, tax_db, on='tax_id', how='left')
    
    # 5. Reorder columns: taxonomy first, then sample counts
    tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
    
    # Find all columns that are sample names
    sample_cols = [col for col in df.columns if col not in ['tax_id', 'raw_taxonomy']]
    
    final_cols = tax_cols + sample_cols
    df_clean = df_merged[final_cols]
    
    # 6. Save the fully populated clean matrix
    df_clean.to_csv(clean_matrix_path, index=False)
    
    print("Data cleaning and taxonomy mapping finished successfully.")
    print(f"Clean matrix saved to: {clean_matrix_path}")

except FileNotFoundError:
    print(f"ERROR: Could not find the taxonomy.tsv file at {tax_db_path}")
    print("Please make sure the path to your database taxonomy file is correct.")

## Creating different tables for analysis

In [ ]:
import pandas as pd
import skbio
import os



# 1. Define paths
matrix_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/4_kma_taxa/otu_abundance_matrix_CLEAN.csv'
meta_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/meta_date_its.csv'

# 2. Load OTU matrix and isolate taxonomy
otu_df = pd.read_csv(matrix_path)
tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']

# Save taxonomy hierarchy separately
taxonomy_df = otu_df[tax_cols].copy().set_index('species')

# 3. Prepare Raw Counts (Genus level)
counts_df = otu_df.drop(columns=[c for c in tax_cols if c != 'genus'])
counts_df = counts_df.groupby('genus').sum(numeric_only=True).T.astype(int)

# 4. Load Metadata and align samples
meta_df = pd.read_csv(meta_path).set_index('sample_title')
common_samples = counts_df.index.intersection(meta_df.index)
counts_df = counts_df.loc[common_samples]
meta_df = meta_df.loc[common_samples]

print("Setup complete!")
print(f"Samples ready: {counts_df.shape[0]}")
print(f"Unique Genera: {counts_df.shape[1]}\n")

# Verification of total reads per sample
sample_totals = counts_df.sum(axis=1)
print("Total reads per sample:")
print(sample_totals.sort_values())

# 5. Rarefy counts for Alpha Diversity
rarefaction_depth = 804000

# Filtered samples based on rarefaction depth 
valid_samples = sample_totals[sample_totals >= rarefaction_depth].index
dropped_samples = set(counts_df.index) - set(valid_samples)

if dropped_samples:
    print(f"\n The samples with less than {rarefaction_depth} reads were removed: {dropped_samples}")

counts_df_valid = counts_df.loc[valid_samples]

counts_rarefied = counts_df_valid.apply(
    lambda x: skbio.stats.subsample_counts(x.values, rarefaction_depth), 
    axis=1, 
    result_type='broadcast'
)
counts_rarefied.columns = counts_df_valid.columns

print(f"\nRarefied matrix created (Depth: {rarefaction_depth} reads/sample).")

In [ ]:
import os

# 1. Define output directory
out_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/5_statistics'
os.makedirs(out_dir, exist_ok=True)

# 2. Filter metadata to match valid rarefied samples
meta_df_valid = meta_df.loc[valid_samples]

# 3. Save filtered metadata
meta_out_path = os.path.join(out_dir, 'metadata_filtered.csv')
meta_df_valid.to_csv(meta_out_path)

# 4. Save rarefied genus abundance counts
counts_out_path = os.path.join(out_dir, 'rarefied_genus_counts.csv')
counts_rarefied.to_csv(counts_out_path)

print(f"Tables successfully saved to:\n{out_dir}")

## Taxonomic composition of the samples

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define INPUT paths (loading the saved tables)
stats_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/5_statistics'
counts_path = os.path.join(stats_dir, 'rarefied_genus_counts.csv')

# Load the counts matrix (index_col=0 keeps the sample names as rows)
counts_rarefied = pd.read_csv(counts_path, index_col=0)

# 2. Define OUTPUT directory for plots
plot_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/6_plots'
os.makedirs(plot_dir, exist_ok=True)

# 3. Convert absolute counts to relative abundance (percentages)
rel_abund = counts_rarefied.div(counts_rarefied.sum(axis=1), axis=0) * 100

# 4. Identify the Top 10 most abundant genera
top_n = 30
top_genera = rel_abund.mean().nlargest(top_n).index

# 5. Create a new dataframe with Top 10 and group the rest into 'Others'
plot_df = rel_abund[top_genera].copy()
plot_df['Others'] = rel_abund.drop(columns=top_genera).sum(axis=1)

# Sort samples alphabetically
plot_df = plot_df.sort_index()

# 6. Plot Stacked Bar Chart
ax = plot_df.plot(kind='bar', stacked=True, figsize=(14, 7), colormap='tab20')

# Formatting the plot
plt.title(f'Taxonomic Composition - Top {top_n} Genera', fontsize=16)
plt.ylabel('Relative Abundance (%)', fontsize=12)
plt.xlabel('Samples', fontsize=12)
plt.legend(title='Genus', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# 7. Save the plot in PNG and SVG
plot_path_png = os.path.join(plot_dir, f'taxonomic_composition_top{top_n}.png')
plot_path_svg = os.path.join(plot_dir, f'taxonomic_composition_top{top_n}.svg')

plt.savefig(plot_path_png, dpi=300, bbox_inches='tight')
plt.savefig(plot_path_svg, bbox_inches='tight')

print(f"Plot successfully saved to:\n{plot_dir}")

# Display the plot in the notebook
plt.show()

## Separation of the samples by groups

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define INPUT paths
stats_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/5_statistics'
counts_path = os.path.join(stats_dir, 'rarefied_genus_counts.csv')
meta_path = os.path.join(stats_dir, 'metadata_filtered.csv')

# 2. Load the tables
counts_rarefied = pd.read_csv(counts_path, index_col=0)
meta_df = pd.read_csv(meta_path, index_col=0)

# 3. IMPORTANT: Define your group column name here!
# Replace 'Treatment' with the exact column name from your metadata
group_col = 'Group'

# Sort metadata by the group column
meta_sorted = meta_df.sort_values(by=[group_col])

# Create a new label combining Group and Sample Name (e.g., "CTL (CTL-3)")
new_labels = meta_sorted[group_col].astype(str) + " (" + meta_sorted.index + ")"

# 4. Convert absolute counts to relative abundance (percentages)
rel_abund = counts_rarefied.div(counts_rarefied.sum(axis=1), axis=0) * 100

# 5. Identify the Top 10 most abundant genera
top_n = 30
top_genera = rel_abund.mean().nlargest(top_n).index

# 6. Create plot dataframe with Top 10 and 'Others'
plot_df = rel_abund[top_genera].copy()
plot_df['Others'] = rel_abund.drop(columns=top_genera).sum(axis=1)

# 7. Reorder plot_df to match the sorted metadata
plot_df = plot_df.loc[meta_sorted.index]
# Apply the new grouped labels to the X-axis
plot_df.index = new_labels

# 8. Define OUTPUT directory
plot_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/6_plots'
os.makedirs(plot_dir, exist_ok=True)

# 9. Plot Stacked Bar Chart
ax = plot_df.plot(kind='bar', stacked=True, figsize=(14, 7), colormap='tab20', width=0.85)

# Formatting the plot
plt.title(f'Taxonomic Composition by Group - Top {top_n} Genera', fontsize=16)
plt.ylabel('Relative Abundance (%)', fontsize=12)
plt.xlabel('Samples (Grouped by Treatment)', fontsize=12)
plt.legend(title='Genus', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# 10. Save the plot
plot_path_png = os.path.join(plot_dir, 'taxonomic_composition_grouped.png')
plot_path_svg = os.path.join(plot_dir, 'taxonomic_composition_grouped.svg')

plt.savefig(plot_path_png, dpi=300, bbox_inches='tight')
plt.savefig(plot_path_svg, bbox_inches='tight')

print(f"Plot successfully saved to:\n{plot_dir}")
plt.show()

## Beta diversity analysis

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from skbio.diversity import beta_diversity
from skbio.stats.ordination import pcoa
from skbio.stats.distance import permanova

# 1. Define INPUT paths
stats_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/5_statistics'
counts_path = os.path.join(stats_dir, 'rarefied_genus_counts.csv')
meta_path = os.path.join(stats_dir, 'metadata_filtered.csv')

# Load the tables
counts_rarefied = pd.read_csv(counts_path, index_col=0)
meta_df = pd.read_csv(meta_path, index_col=0)

# IMPORTANT: Define your group column name here!
group_col = 'Group'

# 2. Calculate Bray-Curtis Distance Matrix
# We use .values to pass the raw numbers, and pass the sample names as ids
bc_dm = beta_diversity("braycurtis", counts_rarefied.values, ids=counts_rarefied.index)

# 3. Perform PCoA (Principal Coordinate Analysis)
bc_pcoa = pcoa(bc_dm)

# Extract the coordinates for PC1 and PC2
pcoa_df = bc_pcoa.samples[['PC1', 'PC2']].copy()
# Join with metadata to get the group labels for coloring the plot
pcoa_df = pcoa_df.join(meta_df[[group_col]])

# Extract the percentage of variance explained by each axis
prop_expl = bc_pcoa.proportion_explained

# 4. Perform PERMANOVA (Statistical test for Beta Diversity)
perm_result = permanova(bc_dm, meta_df, column=group_col, permutations=999)
p_val = perm_result['p-value']
pseudo_f = perm_result['test statistic']

print(f"PERMANOVA Results:")
print(f"Pseudo-F: {pseudo_f:.4f}")
print(f"p-value:  {p_val:.4f}\n")

# 5. Plot the PCoA
plt.figure(figsize=(9, 7))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue=group_col, 
    data=pcoa_df, 
    s=120,          # Marker size
    alpha=0.8,      # Transparency
    palette='Set1', # Color palette
    edgecolor='black'
)

# Formatting the plot
plt.title(f'PCoA - Bray-Curtis Distance\nPERMANOVA: p-value = {p_val:.3f}', fontsize=14)
plt.xlabel(f"PC1 ({prop_expl['PC1']*100:.1f}%)", fontsize=12)
plt.ylabel(f"PC2 ({prop_expl['PC2']*100:.1f}%)", fontsize=12)
plt.legend(title='Group', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.axhline(0, color='black', linewidth=0.8, alpha=0.5) # X axis line
plt.axvline(0, color='black', linewidth=0.8, alpha=0.5) # Y axis line
plt.tight_layout()

# 6. Save the plot
plot_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/6_plots'
os.makedirs(plot_dir, exist_ok=True)

plot_path_png = os.path.join(plot_dir, 'pcoa_bray_curtis.png')
plot_path_svg = os.path.join(plot_dir, 'pcoa_bray_curtis.svg')

plt.savefig(plot_path_png, dpi=300, bbox_inches='tight')
plt.savefig(plot_path_svg, bbox_inches='tight')

print(f"Plot successfully saved to:\n{plot_dir}")

# Display the plot
plt.show()

## Alpha diversity analysis

In [ ]:
import os
import pandas as pd
import skbio
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal

# 1. Define INPUT paths
stats_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/5_statistics'
plot_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/6_plots'

counts_path = os.path.join(stats_dir, 'rarefied_genus_counts.csv')
meta_path = os.path.join(stats_dir, 'metadata_filtered.csv')

# Load the tables
counts_rarefied = pd.read_csv(counts_path, index_col=0)
meta_df = pd.read_csv(meta_path, index_col=0)

# Dynamically extract group names from the sample prefixes
meta_df['Group'] = [str(idx).split('-')[0] for idx in meta_df.index]
groups = list(meta_df['Group'].unique())

# --- NEW: Define the specific order of the groups for the plot ---
control_group = 'CTL'
# This creates a list with 'CTL' first, followed by the rest
group_order = [control_group] + [g for g in groups if g != control_group]
print(f"Plotting order will be: {group_order}")
# -----------------------------------------------------------------

# 2. Calculate Alpha Diversity
meta_df['Observed_Genera'] = skbio.diversity.alpha_diversity('observed_otus', counts_rarefied.values, counts_rarefied.index)
meta_df['Shannon_Index'] = skbio.diversity.alpha_diversity('shannon', counts_rarefied.values, counts_rarefied.index)

# 3. Statistical Test (Global Kruskal-Wallis)
group_data_obs = [meta_df[meta_df['Group'] == g]['Observed_Genera'] for g in group_order]
group_data_sha = [meta_df[meta_df['Group'] == g]['Shannon_Index'] for g in group_order]

stat_obs, p_obs = kruskal(*group_data_obs)
stat_sha, p_sha = kruskal(*group_data_sha)

# 4. Plot Alpha Diversity (Boxplots)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Palette matching the number of groups
palette = sns.color_palette("Set2", len(group_order))

# Plot 1: Richness (Observed Genera)
sns.boxplot(x='Group', y='Observed_Genera', data=meta_df, ax=axes[0], palette=palette, order=group_order, showfliers=False)
sns.stripplot(x='Group', y='Observed_Genera', data=meta_df, ax=axes[0], color='black', alpha=0.7, jitter=True, size=6, order=group_order)
axes[0].set_title(f'Genus Richness\n(Kruskal-Wallis p={p_obs:.3f})', fontsize=14)
axes[0].set_ylabel('Number of Observed Genera', fontsize=12)
axes[0].set_xlabel('Treatment Group', fontsize=12)

# Plot 2: Shannon Diversity
sns.boxplot(x='Group', y='Shannon_Index', data=meta_df, ax=axes[1], palette=palette, order=group_order, showfliers=False)
sns.stripplot(x='Group', y='Shannon_Index', data=meta_df, ax=axes[1], color='black', alpha=0.7, jitter=True, size=6, order=group_order)
axes[1].set_title(f'Shannon Diversity\n(Kruskal-Wallis p={p_sha:.3f})', fontsize=14)
axes[1].set_ylabel('Shannon Index', fontsize=12)
axes[1].set_xlabel('Treatment Group', fontsize=12)

plt.tight_layout()

# 5. Save the plots
plot_path_png = os.path.join(plot_dir, 'alpha_diversity_boxplots.png')
plt.savefig(plot_path_png, dpi=300, bbox_inches='tight')

print(f"Plot successfully saved to:\n{plot_path_png}")
plt.show()

# Bubble plots

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Define paths
matrix_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/4_kma_taxa/otu_abundance_matrix_CLEAN.csv'
stats_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/5_statistics'
plot_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/6_plots'
meta_path = os.path.join(stats_dir, 'metadata_filtered.csv')

os.makedirs(plot_dir, exist_ok=True)
plot_path = os.path.join(plot_dir, 'bubble_abundance_plot.png')

# 2. Load data
otu_df = pd.read_csv(matrix_path)
meta_df = pd.read_csv(meta_path, index_col=0)

# Extract group names from the sample prefixes (e.g., 'CTL-1' -> 'CTL')
meta_df['Group'] = [str(idx).split('-')[0] for idx in meta_df.index]

# 3. Prepare Species Abundance
tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
counts_df = otu_df.drop(columns=[c for c in tax_cols if c != 'species'])
counts_df = counts_df.groupby('species').sum(numeric_only=True).T

# Convert raw counts to Relative Abundance (percentages)
rel_abund = counts_df.div(counts_df.sum(axis=1), axis=0) * 100

# Select the top 20 most abundant species to keep the plot clean
top_species = rel_abund.mean().sort_values(ascending=False).head(20).index
rel_abund_top = rel_abund[top_species]

# 4. Merge with metadata and calculate the mean abundance per Group
common_samples = rel_abund_top.index.intersection(meta_df.index)
rel_abund_top = rel_abund_top.loc[common_samples]
meta_valid = meta_df.loc[common_samples]

# Add group column to abundance data
rel_abund_top['Group'] = meta_valid['Group']

# Calculate the mean relative abundance of each species per group
mean_abund_group = rel_abund_top.groupby('Group').mean()

# 5. Reshape data for seaborn (Melt into long format)
plot_df = mean_abund_group.reset_index().melt(
    id_vars='Group', 
    var_name='Species', 
    value_name='Mean_Abundance'
)

# Remove bubbles where abundance is exactly 0 to clean up the plot
plot_df = plot_df[plot_df['Mean_Abundance'] > 0]

# 6. Plotting
plt.figure(figsize=(8, 10))

# Create the bubble plot
sns.scatterplot(
    data=plot_df,
    x='Group',
    y='Species',
    size='Mean_Abundance',
    hue='Mean_Abundance',
    palette='PuOr_r',       # Purple-Orange colormap
    sizes=(50, 600),        # Minimum and maximum bubble sizes
    edgecolor='gray',
    alpha=0.85
)

# Customization
plt.legend(
    bbox_to_anchor=(1.05, 1), 
    loc='upper left', 
    borderaxespad=0, 
    title="Relative\nAbundance (%)"
)
plt.title('Mean Relative Abundance of Top Species by Group', pad=20, fontsize=14)
plt.xlabel('Experimental Group', fontsize=12)
plt.ylabel('Species', fontsize=12)

# Add a light grid
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()

# Save the plot to the specified directory
plt.savefig(plot_path, dpi=300, bbox_inches='tight')

# --- DISPLAY ON SCREEN ---
plt.show()

# Close the plot to free memory
plt.close()

print(f"Abundance bubble plot successfully saved to:\n{plot_path}")

# Horizontal bar chart

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Define paths
matrix_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/4_kma_taxa/otu_abundance_matrix_CLEAN.csv'
stats_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/5_statistics'
plot_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/6_plots/taxonomy'
meta_path = os.path.join(stats_dir, 'metadata_filtered.csv')

os.makedirs(plot_dir, exist_ok=True)

# 2. Load Data
otu_df = pd.read_csv(matrix_path)
meta_df = pd.read_csv(meta_path, index_col=0)

# Extract group names from the sample prefixes (e.g., 'CTL-1' -> 'CTL')
meta_df['Group'] = [str(idx).split('-')[0] for idx in meta_df.index]
unique_groups = meta_df['Group'].unique()

# 3. Prepare Species Abundance
tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
counts_df = otu_df.drop(columns=[c for c in tax_cols if c != 'species'])
counts_df = counts_df.groupby('species').sum(numeric_only=True).T

# Align samples between abundance matrix and metadata
common_samples = counts_df.index.intersection(meta_df.index)
counts_df = counts_df.loc[common_samples]
meta_df = meta_df.loc[common_samples]

# Convert raw counts to Relative Abundance per sample (0 - 100%)
rel_abund = counts_df.div(counts_df.sum(axis=1), axis=0) * 100
rel_abund['Group'] = meta_df['Group']

# 4. Loop through each group and generate a custom plot
for group in unique_groups:
    print(f"Generating plot for group: {group}...")
    
    # Filter data for the current group and drop the 'Group' text column
    group_data = rel_abund[rel_abund['Group'] == group].drop(columns='Group')
    
    # Calculate the mean relative abundance across samples in this group
    mean_abund = group_data.mean(axis=0)
    
    # Get the top 15 most abundant species for THIS specific group
    top_species = mean_abund.sort_values(ascending=False).head(15)
    
    # Drop any species that might have 0% abundance (just in case)
    top_species = top_species[top_species > 0]
    
    # Sort ascending so the largest bar appears at the TOP of the horizontal plot
    top_species = top_species.sort_values(ascending=True)
    
    # 5. Plotting
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Use viridis colormap, matching the amount of bars
    colors = sns.color_palette("viridis", len(top_species))
    
    bars = ax.barh(top_species.index, top_species.values, color=colors, height=0.6)
    
    # 6. Aesthetic Customizations
    # Remove all borders (spines)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    # Remove the X-axis completely
    ax.set_xticks([])
    
    # Remove Y-axis tick marks, but keep the labels
    ax.tick_params(axis='y', length=0, pad=10, labelsize=12)
    
    # Italicize bacterial names (keeping 'Unclassified' normal)
    for label in ax.get_yticklabels():
        text = label.get_text()
        if text not in ['Unclassified', 'Unknown', 'Não identificado']:
            label.set_fontstyle('italic')
        label.set_color('#4a4a4a') # Dark gray text
        
    # Add the percentage text at the end of each bar
    for bar in bars:
        width = bar.get_width()
        label_x = width + (top_species.max() * 0.02) 
        label_y = bar.get_y() + bar.get_height() / 2
        
        ax.text(
            label_x, 
            label_y, 
            f'{width:.2f} %', 
            va='center', 
            fontweight='bold', 
            fontsize=11,
            color='black'
        )
        
    plt.title(f'Taxonomic Composition - {group}', fontsize=16, pad=20, fontweight='bold')
    plt.tight_layout()
    
    # Save specific plot for the group
    plot_path = os.path.join(plot_dir, f'horizontal_bars_{group}.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    
    # Display the plot in the notebook
    plt.show()
    plt.close()
    
    print(f"Saved: {plot_path}\n")

print("All group plots have been generated!")

## Differential abundance analysis

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

# 1. Define paths
matrix_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/4_kma_taxa/otu_abundance_matrix_CLEAN.csv'
stats_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/5_statistics'
plot_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/its_kma/6_plots'
meta_path = os.path.join(stats_dir, 'metadata_filtered.csv')

os.makedirs(plot_dir, exist_ok=True)

# 2. Load RAW Matrix and Metadata
otu_df = pd.read_csv(matrix_path)
tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
counts_raw = otu_df.drop(columns=[c for c in tax_cols if c != 'genus'])
counts_raw = counts_raw.groupby('genus').sum(numeric_only=True).T
meta_df = pd.read_csv(meta_path, index_col=0)
counts_raw = counts_raw.loc[meta_df.index]

# 3. Dynamically extract group names from the sample prefixes (e.g., 'CTL-1' -> 'CTL')
meta_df['Group'] = [str(idx).split('-')[0] for idx in meta_df.index]
unique_groups = meta_df['Group'].unique()

print(f"Setup complete. Detected groups: {unique_groups}")

# Generate all unique pairs of groups (combinations of 2)
all_pairs = list(combinations(unique_groups, 2))
print(f"Total comparisons to run: {len(all_pairs)}\n")

# Prepare relative abundance with pseudocount
counts_pseudo = counts_raw + 1
rel_abund = counts_pseudo.div(counts_pseudo.sum(axis=1), axis=0)

# 4. Loop through all pairs
for group1, group2 in all_pairs:
    print(f"=========================================")
    print(f"Testing: {group1} vs {group2}")
    
    samples_g1 = meta_df[meta_df['Group'] == group1].index
    samples_g2 = meta_df[meta_df['Group'] == group2].index
    
    df_g1 = rel_abund.loc[samples_g1]
    df_g2 = rel_abund.loc[samples_g2]
    
    results = []
    for genus in rel_abund.columns:
        stat, p_val = mannwhitneyu(df_g1[genus], df_g2[genus], alternative='two-sided')
        mean_g1 = df_g1[genus].mean()
        mean_g2 = df_g2[genus].mean()
        
        # Log2FC > 0 means increased in Group2. Log2FC < 0 means increased in Group1.
        log2fc = np.log2(mean_g2 / mean_g1)
        
        results.append({
            'Genus': genus,
            f'Mean_{group1}': mean_g1,
            f'Mean_{group2}': mean_g2,
            'Log2FC': log2fc,
            'p_value': p_val
        })

    da_df = pd.DataFrame(results).set_index('Genus').dropna()
    
    # FDR Correction
    da_df['FDR_q_value'] = multipletests(da_df['p_value'], method='fdr_bh')[1]
    
    # Filter Significant
    significant_genera = da_df[(da_df['FDR_q_value'] < 0.05) & (da_df['Log2FC'].abs() > 1)].copy()
    significant_genera = significant_genera.sort_values(by='Log2FC', ascending=False)
    
    # Save CSV for this pair
    da_out_path = os.path.join(stats_dir, f'differential_abundance_{group2}_vs_{group1}.csv')
    da_df.to_csv(da_out_path)
    
    # Plot if significant results exist
    if significant_genera.empty:
        print(f"No significant genera found for {group1} vs {group2}.\n")
    else:
        print(f"Found {len(significant_genera)} significant genera! Generating plot...\n")
        
        plt.figure(figsize=(10, max(4, len(significant_genera) * 0.4)))
        
        # Colors: Green if enriched in group2, Red if enriched in group1
        colors = ['#2ca02c' if x > 0 else '#d62728' for x in significant_genera['Log2FC']]
        
        sns.barplot(x=significant_genera['Log2FC'], y=significant_genera.index, palette=colors)
        
        plt.title(f'Differentially Abundant Genera\nGreen = Enriched in {group2} | Red = Enriched in {group1}', fontsize=14)
        plt.xlabel(f'Log2 Fold Change ({group2} / {group1})', fontsize=12)
        plt.ylabel('Genus', fontsize=12)
        plt.axvline(0, color='black', linewidth=1)
        plt.grid(axis='x', linestyle='--', alpha=0.7)
        
        plot_path_png = os.path.join(plot_dir, f'da_barplot_{group2}_vs_{group1}.png')
        plt.savefig(plot_path_png, dpi=300, bbox_inches='tight')
        plt.close() # Close plot to prevent massive notebook scrolling

## Ancombc.R